In [9]:
import pandas as pd
import re

In [2]:
df = pd.read_excel('./eurobarometer_data/eb_105.xlsx', sheet_name='QB7a', header=8)

In [4]:
df

,<<Back to content,Unnamed: 1,UE27\nEU27,BE,BG,CZ,DK,DEW,DE,DEE,...,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE
0,NaN,Total,26415,1014,1015,1036,1001,1220,1515,295,...,502,1012,1004,1025,1052,1054,1009,1003,1009,1026
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,"L’économie (par exemple, la compétitivité)",3205,110,182,127,58,163,215,52,...,55,46,146,89,77,114,169,141,143,88
3,NaN,"Economy (e.g., competitiveness)",0.12,0.11,0.18,0.12,0.06,0.13,0.14,0.18,...,0.11,0.05,0.15,0.09,0.07,0.11,0.17,0.14,0.14,0.09
4,NaN,L'emploi,1999,89,70,44,13,20,24,5,...,37,12,60,55,211,110,19,63,97,23
5,NaN,Employment,0.08,0.09,0.07,0.04,0.01,0.02,0.02,0.02,...,0.07,0.01,0.06,0.05,0.2,0.1,0.02,0.06,0.1,0.02
6,NaN,L'égalité sociale,1401,55,135,60,48,71,92,21,...,27,45,80,54,110,52,67,64,44,40
7,NaN,Social equality,0.05,0.05,0.13,0.06,0.05,0.06,0.06,0.07,...,0.05,0.04,0.08,0.05,0.11,0.05,0.07,0.06,0.04,0.04
8,NaN,L'éducation et la formation,1038,29,18,29,10,42,52,10,...,32,25,39,27,37,70,37,38,29,20
9,NaN,Education and training,0.04,0.03,0.02,0.03,0.01,0.04,0.03,0.03,...,0.06,0.03,0.04,0.03,0.04,0.07,0.04,0.04,0.03,0.02


## VERY IMPORTANT!
Double-check all the french columns. Their presence means that it was not a correct column copied!!!
The condition about BE < 1 fails in come cases, like below:

In [20]:
df = pd.read_excel('./eurobarometer_data/eb_105.xlsx', sheet_name='QB7a', header=8)
df = df.drop(columns={'<<Back to content','UE27\nEU27', 'UE27\\nEU27'},errors='ignore')
num_cols = df.columns.drop('Unnamed: 1')
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df = df[df['BE'] < 1]
df.set_index('Unnamed: 1',inplace=True)
df = df.T
df.index.name = 'country'
df.reset_index(inplace=True)
df.columns.name = None

cleaned_cols = []
for col in df.columns:
    if col == 'country':
        cleaned_cols.append(col)
    else:
        col_str = str(col).lower().strip()
        col_str = col_str.replace("'", "").replace(",", "")
        col_str = col_str.replace(":", "_").replace(" ", "_")
        col_str = re.sub(r'_+', '_', col_str)  # collapses ___ down to _
        cleaned_cols.append(col_str)
df.columns = cleaned_cols

mapping_df = pd.read_csv('mapping_columns.csv')
mapping_df['raw_phrase'] = mapping_df['raw_phrase'].astype(str).str.lower().str.strip()
mapping_df['clean_phrase'] = mapping_df['clean_phrase'].astype(str).str.lower().str.strip()
        
mapping_df['phrase_len'] = mapping_df['raw_phrase'].str.len()
mapping_df = mapping_df.sort_values(by='phrase_len', ascending=False)
column_replacements = dict(zip(mapping_df['raw_phrase'], mapping_df['clean_phrase']))
mapping_df.to_csv('mapping_columns.csv', index=False)

for long_phrase, short_phrase in column_replacements.items():
    df.columns = df.columns.str.replace(long_phrase, short_phrase, regex=False)

df

,country,econmy_(e.g._competitiveness),employment,social_equality,education_and_training,research_and_innvation,climate_and_the_environment,migration,security_and_defence,agriculture,industry_admin_burden,trade_with_countries_outside_the_eu,health,democracy,digital_technlogies_(including_social_media_platforms)_and_digital_transformation,countering_information_manipulation,ne_sait_pas
0,BE,0.11,0.09,0.05,0.03,0.03,0.11,0.15,0.16,0.05,0.04,0.02,0.06,0.04,0.02,0.04,0.0
1,BG,0.18,0.07,0.13,0.02,0.02,0.03,0.05,0.18,0.06,0.04,0.02,0.06,0.02,0.05,0.03,27.0
2,CZ,0.12,0.04,0.06,0.03,0.04,0.03,0.13,0.24,0.03,0.05,0.04,0.06,0.02,0.03,0.03,44.0
3,DK,0.06,0.01,0.05,0.01,0.02,0.25,0.07,0.30,0.04,0.02,0.02,0.04,0.03,0.04,0.03,14.0
4,DEW,0.13,0.02,0.06,0.04,0.03,0.09,0.10,0.21,0.02,0.06,0.02,0.03,0.07,0.04,0.06,17.0
5,DE,0.14,0.02,0.06,0.03,0.03,0.10,0.11,0.20,0.02,0.06,0.02,0.03,0.07,0.04,0.06,17.0
6,DEE,0.18,0.02,0.07,0.03,0.03,0.10,0.13,0.16,0.01,0.06,0.02,0.05,0.06,0.04,0.04,0.0
7,EE,0.20,0.05,0.04,0.07,0.05,0.03,0.04,0.20,0.06,0.05,0.02,0.05,0.02,0.05,0.04,20.0
8,IE,0.09,0.06,0.04,0.05,0.02,0.10,0.13,0.15,0.04,0.02,0.03,0.17,0.03,0.03,0.03,7.0
9,EL,0.27,0.12,0.06,0.05,0.02,0.04,0.07,0.13,0.05,0.01,0.01,0.09,0.05,0.02,0.01,2.0


In [18]:
df

,country,econmy_(e.g._competitiveness),employment,social_equality,education_and_training,research_and_innvation,climate_and_the_environment,migration,security_and_defence,agriculture,industry_admin_burden,trade_with_countries_outside_the_eu,health,democracy,dig_tech_dig_transf,countering_information_manipulation,ne_sait_pas
0,BE,0.11,0.09,0.05,0.03,0.03,0.11,0.15,0.16,0.05,0.04,0.02,0.06,0.04,0.02,0.04,0.0
1,BG,0.18,0.07,0.13,0.02,0.02,0.03,0.05,0.18,0.06,0.04,0.02,0.06,0.02,0.05,0.03,27.0
2,CZ,0.12,0.04,0.06,0.03,0.04,0.03,0.13,0.24,0.03,0.05,0.04,0.06,0.02,0.03,0.03,44.0
3,DK,0.06,0.01,0.05,0.01,0.02,0.25,0.07,0.30,0.04,0.02,0.02,0.04,0.03,0.04,0.03,14.0
4,DEW,0.13,0.02,0.06,0.04,0.03,0.09,0.10,0.21,0.02,0.06,0.02,0.03,0.07,0.04,0.06,17.0
5,DE,0.14,0.02,0.06,0.03,0.03,0.10,0.11,0.20,0.02,0.06,0.02,0.03,0.07,0.04,0.06,17.0
6,DEE,0.18,0.02,0.07,0.03,0.03,0.10,0.13,0.16,0.01,0.06,0.02,0.05,0.06,0.04,0.04,0.0
7,EE,0.20,0.05,0.04,0.07,0.05,0.03,0.04,0.20,0.06,0.05,0.02,0.05,0.02,0.05,0.04,20.0
8,IE,0.09,0.06,0.04,0.05,0.02,0.10,0.13,0.15,0.04,0.02,0.03,0.17,0.03,0.03,0.03,7.0
9,EL,0.27,0.12,0.06,0.05,0.02,0.04,0.07,0.13,0.05,0.01,0.01,0.09,0.05,0.02,0.01,2.0


In [23]:
mapping_df = pd.read_csv('mapping_columns.csv')
mapping_df['raw_phrase'] = mapping_df['raw_phrase'].astype(str).str.lower().str.strip()
mapping_df['clean_phrase'] = mapping_df['clean_phrase'].astype(str).str.lower().str.strip()
        
mapping_df['phrase_len'] = mapping_df['raw_phrase'].str.len()
mapping_df = mapping_df.sort_values(by='phrase_len', ascending=False)
column_replacements = dict(zip(mapping_df['raw_phrase'], mapping_df['clean_phrase']))
mapping_df.to_csv('mapping_columns.csv', index=False)